# 33_gold_marts — Gold Business Marts + Wrapped

Builds all business-facing mart tables from the Gold star schema.
All marts use **full recompute with overwrite** (tens of thousands to low millions of rows,
idempotent by construction; enterprise scale would use CDF-incremental or Materialized Views).

| Table | Business Question |
|---|---|
| `peak_usage_patterns` | Q1: When do users listen? (capacity planning) |
| `subscription_propensity_segments` | Q2: Which free users behave like premium? |
| `content_performance_trends` | Q3: Which genres/artists are rising or falling? |
| `wrapped_user_year` | Consumer Wrapped: one flat row per user per year |
| `user_activity_daily` | Daily per-user activity (feeds cohort + DAU/MAU) |
| `user_cohort_retention` | Monthly cohort retention matrix |
| `skip_by_genre_artist` | Skip-rate & completion by genre/artist/month |
| `audit_pipeline_runs` | Pipeline health by day/layer/table (+ error_message, schema_drift_event_count) |
| `audit_quality_summary` | Quarantine rates by rule (counts only, no raw_record) |
| `audit_freshness` | Latest load timestamp, row count, + catalog coverage per layer/table |

In [0]:
%run ./00_config


## Mart 1: Peak Usage Patterns (Q1 — Capacity Planning)

In [0]:
from datetime import datetime, timezone

_t0 = datetime.now(timezone.utc)
try:
    # Join fact to dim_user to get timezone for local_hour calculation
    df_peak = spark.sql(f"""
        SELECT
            f.hour_of_day                                                AS utc_hour,
            d.day_of_week,
            CASE d.day_of_week WHEN 1 THEN 'Sunday' WHEN 2 THEN 'Monday'
              WHEN 3 THEN 'Tuesday' WHEN 4 THEN 'Wednesday' WHEN 5 THEN 'Thursday'
              WHEN 6 THEN 'Friday' WHEN 7 THEN 'Saturday' END           AS day_name,
            dev.device_category,
            f.country,
            COUNT(*)                                                     AS total_plays,
            SUM(f.minutes_played)                                        AS total_minutes,
            -- local_hour: reconstruct UTC timestamp from date_key + hour_of_day,
            -- then convert to user's timezone.  (The old code converted midnight
            -- — which gives the timezone offset, not the local hour of the play.)
            ANY_VALUE(HOUR(from_utc_timestamp(
                TO_TIMESTAMP(CONCAT(
                    SUBSTR(CAST(f.date_key AS STRING), 1, 4), '-',
                    SUBSTR(CAST(f.date_key AS STRING), 5, 2), '-',
                    SUBSTR(CAST(f.date_key AS STRING), 7, 2), ' ',
                    LPAD(CAST(f.hour_of_day AS STRING), 2, '0'), ':00:00'
                )),
                COALESCE(u.timezone, 'UTC')
            )))                                                          AS local_hour,
            CURRENT_TIMESTAMP()                                          AS load_timestamp
        FROM {table_ref(GOLD['fact_listening'])} f
        JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key = d.date_key
        JOIN {table_ref(GOLD['dim_device'])} dev ON f.device_sk = dev.device_sk
        LEFT JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk = u.user_sk
        WHERE f.minutes_played IS NOT NULL
        GROUP BY f.hour_of_day, d.day_of_week, dev.device_category,
                 f.country, u.timezone
    """)
    n = df_peak.count()
    df_peak.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['peak_usage_patterns'])
    log_run('SILVER_TO_GOLD','gold.peak_usage_patterns','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'peak_usage_patterns: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.peak_usage_patterns','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Mart 2: Subscription Propensity Segments (Q2)

### Discovery-rate strategy

A track play is a *discovery* when the user has **not** played that track
in the preceding 60 days.  Two approaches are valid:

| Approach | Pros | Cons |
|---|---|---|
| Correlated `NOT EXISTS` subquery | Logically clear | O(n²) on large Silver; Spark can't push the correlated predicate into a broadcast |
| `LAG OVER (PARTITION BY user_id, track_id ORDER BY played_at)` | Single-pass window, vectorised | Requires re-sort of the full event history |

**Decision: LAG-based approach** — Silver can have millions of rows; the window
sort is parallelised across Spark partitions whereas a correlated `NOT EXISTS`
forces Spark to execute a nested loop join or a large broadcast.  Both approaches
produce identical results for the 60-day lookback window.

`real_api` rows are excluded from discovery scoring because they carry NULL
`ms_played`/`is_skipped`; including them would corrupt the completion-weighted average.

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    # Compute engagement score components per user
    # discovery_rate: share of plays on a track NOT played by that user in prev 60 days
    # Uses LAG OVER (user_id, track_id ORDER BY played_at): single-pass window,
    # O(n log n) vs O(n^2) for a correlated NOT EXISTS.  Same semantics, same result.
    df_prop = spark.sql(f"""
        WITH current_users AS (
            SELECT user_id, is_premium, country
            FROM {table_ref(GOLD['dim_user'])}
            WHERE is_current = TRUE
        ),
        user_stats AS (
            SELECT
                f.user_sk,
                u.user_id,
                cu.is_premium,
                cu.country,
                SUM(f.minutes_played)                          AS total_minutes,
                SUM(f.play_count)                             AS total_plays,
                COUNT(DISTINCT f.session_id)                   AS session_count,
                COUNT(DISTINCT t.genre)                        AS genre_diversity,
                AVG(CASE WHEN f.is_skipped IS NOT NULL THEN f.is_skipped END) AS skip_rate,
                COUNT(DISTINCT f.device_sk)                    AS raw_device_count
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk=u.user_sk
            JOIN current_users cu ON u.user_id = cu.user_id
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            GROUP BY f.user_sk, u.user_id, cu.is_premium, cu.country
        ),
        -- LAG-based 60-day discovery: a play is new if the previous play of the
        -- same (user, track) pair was >60 days ago OR has never occurred.
        -- real_api rows are excluded (NULL ms_played / is_skipped corrupts averages).
        -- KNOWN LIMITATION: the first play of every track counts as discovery
        -- (prev_played_at IS NULL), so early history inflates the rate.
        discovery_events AS (
            SELECT
                e.user_id,
                e.track_id,
                e.played_at,
                LAG(e.played_at) OVER (
                    PARTITION BY e.user_id, e.track_id
                    ORDER BY e.played_at
                ) AS prev_played_at
            FROM {table_ref(SILVER_EVENTS)} e
            WHERE e.data_source <> 'real_api'
        ),
        discovery AS (
            SELECT user_id,
                   AVG(
                       CASE WHEN prev_played_at IS NULL
                                 OR DATEDIFF(played_at, prev_played_at) > {DISCOVERY_LOOKBACK_DAYS}
                            THEN 1.0 ELSE 0.0 END
                   ) AS discovery_rate
            FROM discovery_events
            GROUP BY user_id
        ),
        vol_norm AS (
            SELECT user_id,
                   (LOG(total_minutes + 1) - mn) / NULLIF(mx - mn, 0) AS volume_norm
            FROM (
                SELECT u2.user_id, SUM(f2.minutes_played) AS total_minutes,
                       MIN(LOG(SUM(f2.minutes_played)+1)) OVER() AS mn,
                       MAX(LOG(SUM(f2.minutes_played)+1)) OVER() AS mx
                FROM {table_ref(GOLD['fact_listening'])} f2
                JOIN {table_ref(GOLD['dim_user'])} u2 ON f2.user_sk=u2.user_sk
                GROUP BY u2.user_id
            ) v
        )
        SELECT
            s.user_id,
            s.is_premium,
            s.country,
            s.total_minutes,
            s.total_plays,
            s.session_count,
            s.genre_diversity,
            COALESCE(s.skip_rate, 0)                             AS skip_rate,
            s.raw_device_count                                     AS distinct_devices,
            -- device_diversity: distinct_devices / max (not entropy)
            s.raw_device_count * 1.0 /
                NULLIF(MAX(s.raw_device_count) OVER(), 0)        AS device_diversity,
            COALESCE(d.discovery_rate, 0)                        AS discovery_rate,
            COALESCE(vn.volume_norm, 0)                          AS volume_norm,
            -- engagement_score = W1*volume + W2*device + W3*(1-skip) + W4*discovery
            {W1_VOLUME}  * COALESCE(vn.volume_norm, 0) +
            {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                NULLIF(MAX(s.raw_device_count) OVER(), 0)) +
            {W3_RETENTION} * (1 - COALESCE(s.skip_rate, 0)) +
            {W4_DISCOVERY} * COALESCE(d.discovery_rate, 0)      AS engagement_score,
            CASE
                WHEN ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                      {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                          NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                      {W3_RETENTION} * (1-COALESCE(s.skip_rate,0)) +
                      {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {PROPENSITY_TIERS[0]} THEN 'High'
                WHEN ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                      {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                          NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                      {W3_RETENTION} * (1-COALESCE(s.skip_rate,0)) +
                      {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {PROPENSITY_TIERS[1]} THEN 'Medium'
                ELSE 'Low'
            END AS propensity_tier,
            (s.is_premium = FALSE OR s.is_premium IS NULL)
                AND ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                     {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                         NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                     {W3_RETENTION} * (1-COALESCE(s.skip_rate,0)) +
                     {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     >= {HIGH_PROPENSITY_THRESHOLD}             AS is_high_conversion_propensity,
            -- churn_risk_flag: premium subscribers whose engagement fell to Low tier
            (s.is_premium = TRUE
                AND ({W1_VOLUME}  * COALESCE(vn.volume_norm,0) +
                     {W2_DEVICE_DIVERSITY} * (s.raw_device_count * 1.0 /
                         NULLIF(MAX(s.raw_device_count) OVER(),0)) +
                     {W3_RETENTION} * (1-COALESCE(s.skip_rate,0)) +
                     {W4_DISCOVERY} * COALESCE(d.discovery_rate,0))
                     < {PROPENSITY_TIERS[1]})                   AS churn_risk_flag,
            CURRENT_TIMESTAMP()                                  AS load_timestamp
        FROM user_stats s
        LEFT JOIN discovery d ON s.user_id = d.user_id
        LEFT JOIN vol_norm  vn ON s.user_id = vn.user_id
    """)
    df_prop = df_prop.withColumn('tier', F.col('propensity_tier'))
    n = df_prop.count()
    df_prop.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['subscription_propensity_segments'])
    flagged = df_prop.filter(F.col('is_high_conversion_propensity')==True).count()
    churn   = df_prop.filter(F.col('churn_risk_flag')==True).count()
    log_run('SILVER_TO_GOLD','gold.subscription_propensity_segments','overwrite',
            _t0,'SUCCESS',rows_read=n,rows_inserted=n)
    print(f'subscription_propensity_segments: {n:,} users | {flagged} High Conversion free users | {churn} churn-risk premium users')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.subscription_propensity_segments','overwrite',
            _t0,'FAILURE',error_message=str(_e))
    raise


## Mart 3: Content Performance Trends (Q3 — Rising/Falling)

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    df_trends = spark.sql(f"""
        WITH monthly AS (
            SELECT
                CAST(DATE_FORMAT(DATE(CONCAT(CAST(d.year AS STRING),'-',
                    LPAD(CAST(d.month AS STRING),2,'0'),'-01')),'yyyyMM') AS INT) AS year_month,
                'genre'                        AS group_type,
                t.genre                        AS group_name,
                SUM(f.minutes_played)          AS monthly_minutes
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY 1, 2, 3
            UNION ALL
            SELECT
                CAST(DATE_FORMAT(DATE(CONCAT(CAST(d.year AS STRING),'-',
                    LPAD(CAST(d.month AS STRING),2,'0'),'-01')),'yyyyMM') AS INT) AS year_month,
                'artist'                       AS group_type,
                t.artist_name                  AS group_name,
                SUM(f.minutes_played)          AS monthly_minutes
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY 1, 2, 3
        ),
        totals AS (
            SELECT year_month, group_type,
                   SUM(monthly_minutes) AS total_minutes
            FROM monthly GROUP BY year_month, group_type
        ),
        with_share AS (
            SELECT m.year_month, m.group_type, m.group_name, m.monthly_minutes,
                   m.monthly_minutes / NULLIF(t.total_minutes, 0) AS monthly_share
            FROM monthly m JOIN totals t
              ON m.year_month=t.year_month AND m.group_type=t.group_type
        )
        SELECT
            year_month, group_type, group_name, monthly_minutes,
            monthly_share,
            monthly_share - LAG(monthly_share) OVER (
                PARTITION BY group_type, group_name ORDER BY year_month
            ) AS mom_delta,
            CURRENT_TIMESTAMP() AS load_timestamp
        FROM with_share
    """)
    n = df_trends.count()
    df_trends.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['content_performance_trends'])
    log_run('SILVER_TO_GOLD','gold.content_performance_trends','overwrite',
            _t0,'SUCCESS',rows_read=n,rows_inserted=n)
    print(f'content_performance_trends: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.content_performance_trends','overwrite',
            _t0,'FAILURE',error_message=str(_e))
    raise


## Mart 4: Wrapped User Year

One row per user per year, shaped to drive the `Mixtape_Wrapped_storyboard.html`
storyboard. Replaces the earlier flat `top_artist_1..5` columns with the nested
fields the storyboard actually consumes: top lists, biggest day, most-replayed,
longest streak, 12×31 calendar, 24-hour curve, per-artist monthly race and top
tracks, skip breakdown, and discovery rate.

Mapping notes vs. the storyboard's client-side `D` object:

- `total_minutes` uses **all** plays; `total_plays`, top lists and the 24-hour
  curve use **non-skipped** plays (`is_skipped = 0`), mirroring the storyboard's
  "skip = ms < 30s" gating.
- `month` values are 1-12 (SQL convention); the storyboard indexes months 0-11,
  so a thin adapter subtracts 1 when serialising a row to JSON.
- `calendar` is 12 strings of 31 chars: `1` = active, `2` = streak, `0` = off,
  `-` = not a real day in that month.
- `discovery_rate` is the authoritative 60-day LAG value from the propensity mart.
- `real_api` rows are excluded (NULL `ms_played` / `is_skipped`), as elsewhere.

Assembly runs on the driver (`collect()` of small per-user-year aggregates), which
is safe here: `wrapped_user_year` is one row per user-year, not fact-sized.


In [0]:
from datetime import timedelta
import collections, calendar

_t0 = datetime.now(timezone.utc)
try:
    # Base view: one play per row for the Wrapped year (fact joined to dims).
    # `minutes_played IS NOT NULL` drops real_api rows (NULL ms_played), matching
    # the reconciliation rule that excludes real_api from skip/completion features.
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW wrapped_base AS
        SELECT
            user_id,
            YEAR(local_ts)  AS year,
            MONTH(local_ts) AS month,
            DAY(local_ts)   AS day_of_month,
            DATE(local_ts)  AS date,
            HOUR(local_ts)  AS hour_of_day,
            artist_name,
            track_name,
            genre,
            minutes_played,
            is_skipped,
            track_sk
        FROM (
            SELECT
                u.user_id,
                -- Reconstruct UTC timestamp from date_key + hour_of_day, convert
                -- to user's local timezone.  Old code used UTC date/hour, so peak
                -- hour, biggest day and the calendar were shifted by the tz offset.
                from_utc_timestamp(
                    TO_TIMESTAMP(CONCAT(
                        SUBSTR(CAST(f.date_key AS STRING), 1, 4), '-',
                        SUBSTR(CAST(f.date_key AS STRING), 5, 2), '-',
                        SUBSTR(CAST(f.date_key AS STRING), 7, 2), ' ',
                        LPAD(CAST(f.hour_of_day AS STRING), 2, '0'), ':00:00'
                    )),
                    COALESCE(u.timezone, 'UTC')
                ) AS local_ts,
                t.artist_name,
                t.track_name,
                t.genre,
                f.minutes_played,
                f.is_skipped,
                f.track_sk
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])}  u ON f.user_sk = u.user_sk
            JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk = t.track_sk
            WHERE f.minutes_played IS NOT NULL
        ) base
    """)

    # ---- Headline totals (all plays) ----
    df_totals = spark.sql("""
        SELECT
            user_id, year,
            SUM(minutes_played)                                            AS total_minutes,
            COUNT(*)                                                       AS total_events,
            SUM(CASE WHEN is_skipped = 1 THEN 1 ELSE 0 END)                AS skip_count,
            SUM(CASE WHEN is_skipped = 1 THEN 0 ELSE 1 END)                AS total_plays,
            COUNT(DISTINCT CASE WHEN is_skipped = 0 THEN track_sk END)     AS distinct_tracks,
            COUNT(DISTINCT CASE WHEN is_skipped = 0 THEN artist_name END)  AS distinct_artists,
            COUNT(DISTINCT date)                                           AS active_days,
            DATEDIFF(MAX(date), MIN(date)) + 1                             AS span_days
        FROM wrapped_base
        GROUP BY user_id, year
    """)

    # ---- 24-hour distribution (non-skipped) ----
    hour_case = ",\n".join(
        f"            SUM(CASE WHEN is_skipped = 0 AND hour_of_day = {h} THEN 1 ELSE 0 END) AS h{h}"
        for h in range(24))
    df_hours = spark.sql(f"""
        SELECT user_id, year,
{hour_case}
        FROM wrapped_base GROUP BY user_id, year
    """).withColumn('hours', F.array([F.col(f'h{h}') for h in range(24)])) \
        .select('user_id', 'year', 'hours')

    # ---- Aggregates for top lists, daily patterns, and skips ----
    df_artist = spark.sql("""
        SELECT user_id, year, artist_name,
               COUNT(*)                 AS plays,
               SUM(minutes_played)      AS minutes,
               COUNT(DISTINCT track_sk) AS nt
        FROM wrapped_base WHERE is_skipped = 0
        GROUP BY user_id, year, artist_name
    """)
    df_artist_month = spark.sql("""
        SELECT user_id, year, artist_name, month, COUNT(*) AS plays
        FROM wrapped_base WHERE is_skipped = 0
        GROUP BY user_id, year, artist_name, month
    """)
    df_track = spark.sql("""
        SELECT user_id, year, track_name, artist_name,
               COUNT(*)            AS plays,
               SUM(minutes_played) AS minutes
        FROM wrapped_base WHERE is_skipped = 0
        GROUP BY user_id, year, track_name, artist_name
    """)
    df_genre = spark.sql("""
        SELECT user_id, year, genre, COUNT(*) AS plays
        FROM wrapped_base
        WHERE is_skipped = 0 AND genre IS NOT NULL AND genre <> 'unclassified'
        GROUP BY user_id, year, genre
    """)
    df_day = spark.sql("""
        SELECT user_id, year, month, day_of_month, date,
               SUM(minutes_played) AS day_minutes,
               SUM(CASE WHEN is_skipped = 0 THEN 1 ELSE 0 END) AS day_plays
        FROM wrapped_base GROUP BY user_id, year, month, day_of_month, date
    """)
    df_day_artist = spark.sql("""
        SELECT user_id, year, date, artist_name, COUNT(*) AS plays
        FROM wrapped_base WHERE is_skipped = 0
        GROUP BY user_id, year, date, artist_name
    """)
    df_day_track = spark.sql("""
        SELECT user_id, year, date, month, day_of_month, track_name, artist_name,
               COUNT(*) AS plays
        FROM wrapped_base WHERE is_skipped = 0
        GROUP BY user_id, year, date, month, day_of_month, track_name, artist_name
    """)
    df_skip_artist = spark.sql("""
        SELECT user_id, year, artist_name,
               COUNT(*)                                          AS plays,
               SUM(CASE WHEN is_skipped = 1 THEN 1 ELSE 0 END)   AS skips
        FROM wrapped_base GROUP BY user_id, year, artist_name
    """)
    df_disc = spark.sql(f"""
        SELECT user_id, discovery_rate
        FROM {table_ref(GOLD['subscription_propensity_segments'])}
    """)

    # ---- Assemble on the driver. wrapped_user_year is one row per user-year
    # (tiny), so collecting these small aggregates is not fact-sized and is far
    # clearer than nesting ARRAY<ARRAY<STRUCT<...>>> in SQL.
    totals = [r.asDict() for r in df_totals.collect()]
    hours  = {(r.user_id, r.year): r.hours for r in df_hours.collect()}
    disc   = {r.user_id: (r.discovery_rate or 0.0) for r in df_disc.collect()}

    def group(rows):
        g = collections.defaultdict(list)
        for r in rows:
            g[(r.user_id, r.year)].append(r)
        return g

    artist_g       = group(df_artist.collect())
    artist_month_g = group(df_artist_month.collect())
    track_g        = group(df_track.collect())
    genre_g        = group(df_genre.collect())
    day_g          = group(df_day.collect())
    day_artist_g   = group(df_day_artist.collect())
    day_track_g    = group(df_day_track.collect())
    skip_g         = group(df_skip_artist.collect())

    out = []
    for t in totals:
        key = (t['user_id'], t['year'])
        arts = sorted(artist_g.get(key, []), key=lambda r: (-r.plays, r.artist_name))
        top5 = arts[:5]
        trks = sorted(track_g.get(key, []), key=lambda r: (-r.minutes, r.track_name, r.artist_name))
        gens = genre_g.get(key, [])
        days = day_g.get(key, [])
        darts = day_artist_g.get(key, [])
        dtrks = day_track_g.get(key, [])
        sarts = skip_g.get(key, [])

        non_skipped = t['total_plays'] or 1

        top_artists = [
            {'name': r.artist_name, 'plays': int(r.plays),
             'minutes': float(r.minutes or 0), 'distinct_tracks': int(r.nt or 0)}
            for r in top5]

        top_tracks = [
            {'title': r.track_name, 'artist': r.artist_name,
             'plays': int(r.plays), 'minutes': float(r.minutes or 0)}
            for r in trks[:10]]

        top_genres = [
            {'name': r.genre, 'pct': float(round(r.plays / non_skipped * 100))}
            for r in sorted(gens, key=lambda r: -r.plays)[:5]]

        # Per-artist monthly + top tracks, aligned to top_artists order
        am = collections.defaultdict(int)
        for r in artist_month_g.get(key, []):
            am[(r.artist_name, r.month)] = int(r.plays)
        trk_by_artist = collections.defaultdict(list)
        for r in track_g.get(key, []):
            trk_by_artist[r.artist_name].append(r)
        monthly_plays = []
        by_artist_tracks = []
        for r in top5:
            monthly_plays.append([int(am[(r.artist_name, m)]) for m in range(1, 13)])
            at = sorted(trk_by_artist.get(r.artist_name, []), key=lambda x: -x.plays)[:5]
            by_artist_tracks.append([{'title': x.track_name, 'plays': int(x.plays)} for x in at])

        # Biggest day (max minutes) + its top artist
        biggest = None
        if days:
            bd = max(days, key=lambda r: r.day_minutes or 0)
            bd_arts = sorted([x for x in darts if x.date == bd.date], key=lambda x: -x.plays)
            biggest = {
                'month': int(bd.month), 'day': int(bd.day_of_month),
                'minutes': round(bd.day_minutes or 0),
                'artist': bd_arts[0].artist_name if bd_arts else None,
                'plays': int(bd.day_plays or 0),
                'artist_plays': int(bd_arts[0].plays) if bd_arts else 0}

        # Most-replayed track in a single day
        most_replayed = None
        if dtrks:
            mr = max(dtrks, key=lambda r: r.plays)
            most_replayed = {
                'month': int(mr.month), 'day': int(mr.day_of_month),
                'title': mr.track_name, 'artist': mr.artist_name,
                'times': int(mr.plays)}

        # Longest consecutive-day streak + 12x31 calendar
        active_dates = sorted({r.date for r in days})
        streak_dates = set()
        best_len = 0; best_start = None; best_end = None
        run_len = 0; run_start = None; prev = None
        for dt in active_dates:
            if prev is not None and (dt - prev).days == 1:
                run_len += 1
            else:
                run_len = 1
                run_start = dt
            if run_len > best_len:
                best_len = run_len
                best_start = run_start
                best_end = dt
            prev = dt
        streak = None
        if best_start is not None:
            d = best_start
            while d <= best_end:
                streak_dates.add(d)
                d = d + timedelta(days=1)
            streak = {
                'length': best_len,
                'start_month': best_start.month, 'start_day': best_start.day,
                'end_month': best_end.month, 'end_day': best_end.day}

        cal = []
        for m in range(1, 13):
            ndays = calendar.monthrange(t['year'], m)[1]
            chars = []
            for d in range(1, 32):
                if d > ndays:
                    chars.append('-')
                else:
                    dt = datetime(t['year'], m, d).date()
                    if dt in streak_dates:
                        chars.append('2')
                    elif dt in active_dates:
                        chars.append('1')
                    else:
                        chars.append('0')
            cal.append(''.join(chars))

        # Skip worst artists (>= 80 plays, top 3 by skip rate)
        skip_worst = []
        for r in sorted([x for x in sarts if x.plays >= 80],
                        key=lambda x: -((x.skips or 0) / (x.plays or 1)))[:3]:
            skip_worst.append({'name': r.artist_name,
                               'pct': float(round((r.skips or 0) / (r.plays or 1) * 100)),
                               'plays': int(r.plays)})

        hrs = hours.get(key) or [0] * 24
        peak_hour = int(max(range(24), key=lambda h: hrs[h])) if any(hrs) else None

        out.append({
            'user_id': t['user_id'],
            'year': t['year'],
            'total_minutes': float(t['total_minutes'] or 0),
            'total_plays': int(t['total_plays'] or 0),
            'total_events': int(t['total_events'] or 0),
            'distinct_tracks': int(t['distinct_tracks'] or 0),
            'distinct_artists': int(t['distinct_artists'] or 0),
            'active_days': int(t['active_days'] or 0),
            'span_days': int(t['span_days'] or 0),
            'skip_pct': float(round((t['skip_count'] or 0) / (t['total_events'] or 1) * 100, 1)),
            'skip_count': int(t['skip_count'] or 0),
            'skip_worst_artists': skip_worst,
            'biggest_day': biggest,
            'most_replayed': most_replayed,
            'longest_streak': streak,
            'calendar': cal,
            'hours': list(hrs),
            'top_artists': top_artists,
            'top_tracks': top_tracks,
            'top_genres': top_genres,
            'by_artist_tracks': by_artist_tracks,
            'monthly_plays': monthly_plays,
            'peak_hour': peak_hour,
            'discovery_rate': float(disc.get(t['user_id']) or 0.0),
        })

    schema = StructType([
        StructField('user_id', StringType(), False),
        StructField('year', IntegerType(), False),
        StructField('total_minutes', DoubleType()),
        StructField('total_plays', LongType()),
        StructField('total_events', LongType()),
        StructField('distinct_tracks', IntegerType()),
        StructField('distinct_artists', IntegerType()),
        StructField('active_days', IntegerType()),
        StructField('span_days', IntegerType()),
        StructField('skip_pct', DoubleType()),
        StructField('skip_count', LongType()),
        StructField('skip_worst_artists', ArrayType(StructType([
            StructField('name', StringType()),
            StructField('pct', DoubleType()),
            StructField('plays', LongType()),
        ]))),
        StructField('biggest_day', StructType([
            StructField('month', IntegerType()),
            StructField('day', IntegerType()),
            StructField('minutes', DoubleType()),
            StructField('artist', StringType()),
            StructField('plays', LongType()),
            StructField('artist_plays', LongType()),
        ])),
        StructField('most_replayed', StructType([
            StructField('month', IntegerType()),
            StructField('day', IntegerType()),
            StructField('title', StringType()),
            StructField('artist', StringType()),
            StructField('times', LongType()),
        ])),
        StructField('longest_streak', StructType([
            StructField('length', IntegerType()),
            StructField('start_month', IntegerType()),
            StructField('start_day', IntegerType()),
            StructField('end_month', IntegerType()),
            StructField('end_day', IntegerType()),
        ])),
        StructField('calendar', ArrayType(StringType())),
        StructField('hours', ArrayType(IntegerType())),
        StructField('top_artists', ArrayType(StructType([
            StructField('name', StringType()),
            StructField('plays', LongType()),
            StructField('minutes', DoubleType()),
            StructField('distinct_tracks', IntegerType()),
        ]))),
        StructField('top_tracks', ArrayType(StructType([
            StructField('title', StringType()),
            StructField('artist', StringType()),
            StructField('plays', LongType()),
            StructField('minutes', DoubleType()),
        ]))),
        StructField('top_genres', ArrayType(StructType([
            StructField('name', StringType()),
            StructField('pct', DoubleType()),
        ]))),
        StructField('by_artist_tracks', ArrayType(ArrayType(StructType([
            StructField('title', StringType()),
            StructField('plays', LongType()),
        ])))),
        StructField('monthly_plays', ArrayType(ArrayType(IntegerType()))),
        StructField('peak_hour', IntegerType()),
        StructField('discovery_rate', DoubleType()),
    ])

    df_out = spark.createDataFrame(out, schema).withColumn('load_timestamp', F.current_timestamp())
    n = df_out.count()
    df_out.write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
        .saveAsTable(GOLD['wrapped_user_year'])
    log_run('SILVER_TO_GOLD', 'gold.wrapped_user_year', 'overwrite', _t0, 'SUCCESS',
            rows_read=n, rows_inserted=n)
    print(f'wrapped_user_year: {n:,} user-year rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD', 'gold.wrapped_user_year', 'overwrite', _t0, 'FAILURE',
            error_message=str(_e))
    raise


## Mart 5: User Activity Daily (feeds DAU/MAU + cohort)

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    df_daily = spark.sql(f"""
        WITH current_users AS (
            SELECT user_id, is_premium
            FROM {table_ref(GOLD['dim_user'])}
            WHERE is_current = TRUE
        ),
        user_day AS (
            SELECT
                u.user_id,
                d.date                                          AS activity_date,
                cu.is_premium,
                COUNT(f.event_id)                               AS total_plays,
                SUM(f.minutes_played)                           AS total_minutes,
                COUNT(DISTINCT f.track_sk)                      AS distinct_tracks,
                COUNT(DISTINCT t.genre)                         AS distinct_genres,
                SUM(CASE WHEN f.is_skipped=1 THEN 1 ELSE 0 END) AS skip_count,
                SUM(CASE WHEN f.is_skipped=1 THEN 1 ELSE 0 END) /
                    NULLIF(COUNT(CASE WHEN f.is_skipped IS NOT NULL THEN 1 END), 0)
                                                                AS skip_rate
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u   ON f.user_sk=u.user_sk
            JOIN current_users cu ON u.user_id = cu.user_id
            JOIN {table_ref(GOLD['dim_date'])} d   ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_track'])} t  ON f.track_sk=t.track_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY u.user_id, d.date, cu.is_premium
        ),
        device_counts AS (
            SELECT
                u.user_id,
                d.date AS activity_date,
                dev.device_category,
                COUNT(*) AS device_plays
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk=u.user_sk
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            JOIN {table_ref(GOLD['dim_device'])} dev ON f.device_sk=dev.device_sk
            WHERE f.minutes_played IS NOT NULL
            GROUP BY u.user_id, d.date, dev.device_category
        ),
        primary AS (
            SELECT user_id, activity_date, device_category AS primary_device
            FROM (
                SELECT user_id, activity_date, device_category,
                       ROW_NUMBER() OVER (PARTITION BY user_id, activity_date
                                          ORDER BY device_plays DESC) AS rn
                FROM device_counts
            ) WHERE rn = 1
        )
        SELECT ud.user_id, ud.activity_date, ud.is_premium,
               ud.total_plays, ud.total_minutes, ud.distinct_tracks,
               ud.distinct_genres, ud.skip_count, ud.skip_rate,
               p.primary_device,
               CURRENT_TIMESTAMP() AS load_timestamp
        FROM user_day ud
        LEFT JOIN primary p ON ud.user_id = p.user_id
                           AND ud.activity_date = p.activity_date
    """)
    n = df_daily.count()
    df_daily.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['user_activity_daily'])
    log_run('SILVER_TO_GOLD','gold.user_activity_daily','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'user_activity_daily: {n:,} user-day rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.user_activity_daily','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Mart 6: User Cohort Retention

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    df_cohort = spark.sql(f"""
        WITH first_play AS (
            SELECT
                u.user_id,
                CAST(DATE_FORMAT(MIN(d.date), 'yyyyMM') AS INT) AS cohort_month
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u  ON f.user_sk=u.user_sk
            JOIN {table_ref(GOLD['dim_date'])} d  ON f.date_key=d.date_key
            GROUP BY u.user_id
        ),
        monthly_active AS (
            SELECT
                u.user_id,
                CAST(DATE_FORMAT(d.date, 'yyyyMM') AS INT) AS active_month
            FROM {table_ref(GOLD['fact_listening'])} f
            JOIN {table_ref(GOLD['dim_user'])} u ON f.user_sk=u.user_sk
            JOIN {table_ref(GOLD['dim_date'])} d ON f.date_key=d.date_key
            GROUP BY u.user_id, CAST(DATE_FORMAT(d.date, 'yyyyMM') AS INT)
        ),
        cohort_activity AS (
            SELECT
                fp.cohort_month,
                ma.active_month,
                -- Approximate months offset: year*12+month arithmetic
                (INT(SUBSTRING(CAST(ma.active_month AS STRING), 1, 4)) * 12 +
                 INT(SUBSTRING(CAST(ma.active_month AS STRING), 5, 2)))
                - (INT(SUBSTRING(CAST(fp.cohort_month AS STRING), 1, 4)) * 12 +
                   INT(SUBSTRING(CAST(fp.cohort_month AS STRING), 5, 2))) AS offset_months,
                COUNT(DISTINCT fp.user_id)                               AS retained_users
            FROM first_play fp
            JOIN monthly_active ma ON fp.user_id = ma.user_id
            GROUP BY fp.cohort_month, ma.active_month
        ),
        cohort_sizes AS (
            SELECT cohort_month, COUNT(DISTINCT user_id) AS cohort_size
            FROM first_play GROUP BY cohort_month
        )
        SELECT
            ca.cohort_month,
            ca.offset_months,
            cs.cohort_size,
            ca.retained_users,
            ca.retained_users / NULLIF(cs.cohort_size, 0) AS retention_rate,
            CURRENT_TIMESTAMP()                           AS load_timestamp
        FROM cohort_activity ca
        JOIN cohort_sizes cs ON ca.cohort_month = cs.cohort_month
        ORDER BY ca.cohort_month, ca.offset_months
    """)
    n = df_cohort.count()
    df_cohort.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['user_cohort_retention'])
    log_run('SILVER_TO_GOLD','gold.user_cohort_retention','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'user_cohort_retention: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.user_cohort_retention','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Mart 7: Skip by Genre / Artist

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    df_skip = spark.sql(f"""
        SELECT
            t.genre,
            t.artist_name,
            CAST(DATE_FORMAT(DATE(CONCAT(
                CAST(d.year AS STRING), '-',
                LPAD(CAST(d.month AS STRING), 2, '0'), '-01'
            )), 'yyyyMM') AS INT)                              AS year_month,
            COUNT(f.event_id)                                  AS total_plays,
            SUM(CASE WHEN f.is_skipped=1 THEN 1 ELSE 0 END)   AS skip_count,
            SUM(CASE WHEN f.is_skipped=1 THEN 1 ELSE 0 END) /
                NULLIF(COUNT(CASE WHEN f.is_skipped IS NOT NULL THEN 1 END), 0)
                                                              AS skip_rate,
            AVG(CASE WHEN f.data_source <> 'real_api' THEN f.completion_rate END) AS avg_completion_rate,
            SUM(f.minutes_played)                              AS total_minutes,
            CURRENT_TIMESTAMP()                                AS load_timestamp
        FROM {table_ref(GOLD['fact_listening'])} f
        JOIN {table_ref(GOLD['dim_track'])} t ON f.track_sk=t.track_sk
        JOIN {table_ref(GOLD['dim_date'])}  d ON f.date_key=d.date_key
        WHERE f.minutes_played IS NOT NULL
        GROUP BY t.genre, t.artist_name, d.year, d.month
    """)
    n = df_skip.count()
    df_skip.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['skip_by_genre_artist'])
    log_run('SILVER_TO_GOLD','gold.skip_by_genre_artist','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'skip_by_genre_artist: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.skip_by_genre_artist','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 1: Pipeline Runs

Adds `error_message` (most-recent error per table) and `schema_drift_event_count`
(runs where `error_message` mentions `SCHEMA_DRIFT`) on top of the base aggregates.

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    df_runs = spark.sql(f"""
        SELECT
            DATE(start_time)      AS run_date,
            layer,
            target_table,
            COUNT(*)              AS run_count,
            SUM(CASE WHEN status='SUCCESS' THEN 1 ELSE 0 END) AS success_count,
            SUM(CASE WHEN status='FAILURE' THEN 1 ELSE 0 END) AS failure_count,
            SUM(COALESCE(rows_read,0))     AS rows_read_sum,
            SUM(COALESCE(rows_inserted,0)
                + COALESCE(rows_updated,0))  AS rows_out_sum,
            PERCENTILE_APPROX(duration_sec,0.5) AS duration_p50,
            MAX(duration_sec)               AS duration_max,
            -- Most recent error text for this (run_date, layer, target_table) group.
            -- NULL when all runs succeeded.  Surfaces SCHEMA_DRIFT annotations
            -- written by 10_bronze_ingest when rescued columns are detected.
            MAX_BY(error_message, end_time) AS error_message,
            -- Count of runs where upstream schema drift was detected.
            SUM(CASE WHEN error_message LIKE '%SCHEMA_DRIFT%' THEN 1 ELSE 0 END)
                                            AS schema_drift_event_count,
            CURRENT_TIMESTAMP()             AS load_timestamp
        FROM {table_ref(EXECUTION_LOG)}
        GROUP BY DATE(start_time), layer, target_table
    """)
    n = df_runs.count()
    df_runs.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_pipeline_runs'])
    log_run('SILVER_TO_GOLD','gold.audit_pipeline_runs','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_pipeline_runs: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_pipeline_runs','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 2: Quality Summary (counts only — no raw_record)

Bronze: rejections per `(partition, reason)` with `quarantine_rate` and `above_alert`
(2% threshold). Silver: the quarantine table carries no `_partition_dt`, but every
quarantined event still lives in `bronze.events` (immutable), so rows are attributed
to their source partition by joining on `event_id` — matching `local_gold_build.py`,
which keeps `_partition_dt` through the silver build.

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    # Bronze quarantine counts by _reason
    df_bq = spark.sql(f"""
        SELECT _partition_dt AS partition_dt, 'bronze' AS layer, _reason AS dq_rule,
               COUNT(*) AS rejection_count
        FROM {table_ref(BRONZE_EVENTS_QUARANTINE)}
        GROUP BY _partition_dt, _reason
    """)
    # Silver quarantine counts by _dq_rule, attributed to the source partition via
    # bronze.events (silver.events_quarantine has no _partition_dt column, but every
    # quarantined row still exists in the immutable bronze.events).
    df_sq = spark.sql(f"""
        SELECT e._partition_dt AS partition_dt, 'silver' AS layer, q._dq_rule AS dq_rule,
               COUNT(DISTINCT q.event_id) AS rejection_count
        FROM {table_ref(SILVER_EVENTS_QUARANTINE)} q
        JOIN {table_ref(BRONZE_EVENTS)} e ON q.event_id = e.event_id
        GROUP BY e._partition_dt, q._dq_rule
    """)
    # Total events per partition for bronze rate — DISTINCT because bronze is
    # append-only and holds re-ingested duplicate event_ids across batches.
    df_bq_tot = spark.sql(f"""
        SELECT _partition_dt, COUNT(DISTINCT event_id) AS total
        FROM {table_ref(BRONZE_EVENTS)} GROUP BY _partition_dt
    """)

    df_qual = df_bq.join(df_bq_tot, df_bq['partition_dt']==df_bq_tot['_partition_dt'],'left') \
        .withColumn('total_count', F.coalesce(F.col('total'), F.lit(0))) \
        .withColumn('quarantine_rate', F.col('rejection_count') / F.greatest(F.col('total_count'),F.lit(1))) \
        .withColumn('above_alert', F.col('quarantine_rate') > F.lit(QUARANTINE_ALERT_RATE)) \
        .withColumn('load_timestamp', F.current_timestamp()) \
        .select('partition_dt','layer','dq_rule','rejection_count','total_count',
                'quarantine_rate','above_alert','load_timestamp') \
        .union(
            df_sq.join(df_bq_tot, df_sq['partition_dt']==df_bq_tot['_partition_dt'],'left')
                .withColumn('total_count', F.coalesce(F.col('total'), F.lit(0)))
                .withColumn('quarantine_rate', F.col('rejection_count') / F.greatest(F.col('total_count'),F.lit(1)))
                .withColumn('above_alert', F.col('quarantine_rate') > F.lit(QUARANTINE_ALERT_RATE))
                .withColumn('load_timestamp', F.current_timestamp())
                .select('partition_dt','layer','dq_rule','rejection_count','total_count',
                        'quarantine_rate','above_alert','load_timestamp')
        )

    n = df_qual.count()
    df_qual.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_quality_summary'])
    log_run('SILVER_TO_GOLD','gold.audit_quality_summary','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_quality_summary: {n:,} rows')
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_quality_summary','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Audit Mart 3: Data Freshness + Catalog Coverage

Adds two catalog-coverage metrics to catalog tables:
- `unclassified_track_share`: share of rows with `genre = 'unclassified'`.
- `inferred_track_count`: count of `is_inferred = true` rows in `gold.dim_track`.

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    tables = [
        ('bronze', 'bronze.events',           BRONZE_EVENTS,          '_partition_dt', False, False),
        ('silver', 'silver.events',            SILVER_EVENTS,          '_partition_dt', False, False),
        ('silver', 'silver.users',             SILVER_USERS,           None,            False, False),
        ('silver', 'silver.tracks',            SILVER_TRACKS,          None,            True,  False),
        ('gold',   'gold.fact_listening',      GOLD['fact_listening'], None,            False, False),
        ('gold',   'gold.dim_user',            GOLD['dim_user'],       None,            False, False),
        ('gold',   'gold.dim_track',           GOLD['dim_track'],      None,            True,  True),
    ]
    rows = []
    for layer, tname, tref, partcol, has_genre, has_inferred in tables:
        df_t = spark.table(tref)
        cnt = df_t.count()
        max_ts = df_t.agg(F.max('load_timestamp')).first()[0]
        max_part = df_t.agg(F.max(partcol)).first()[0] if partcol else None
        # Catalog coverage: unclassified share
        if has_genre and cnt > 0:
            unclassified_share = df_t.agg(
                (F.sum(F.when(F.col('genre') == 'unclassified', 1).otherwise(0)) / F.count('*'))
                .alias('s')
            ).first()['s']
        else:
            unclassified_share = None
        # Inferred track count (dim_track only)
        if has_inferred:
            inferred_count = df_t.filter(F.col('is_inferred') == True).count()
        else:
            inferred_count = None
        rows.append((layer, tname, max_part, max_ts, cnt, unclassified_share, inferred_count))

    from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType, LongType, DoubleType
    schema = StructType([
        StructField('layer', StringType()),
        StructField('table_name', StringType()),
        StructField('latest_partition', DateType()),
        StructField('latest_load_ts', TimestampType()),
        StructField('row_count', LongType()),
        StructField('unclassified_track_share', DoubleType()),
        StructField('inferred_track_count', LongType()),
    ])
    df_fresh = spark.createDataFrame(rows, schema).withColumn('load_timestamp', F.current_timestamp())
    n = df_fresh.count()
    df_fresh.write.format('delta').mode('overwrite').option('overwriteSchema','true') \
        .saveAsTable(GOLD['audit_freshness'])
    log_run('SILVER_TO_GOLD','gold.audit_freshness','overwrite',_t0,'SUCCESS',
            rows_read=n,rows_inserted=n)
    print(f'audit_freshness: {n:,} rows')
    display(df_fresh)
except Exception as _e:
    log_run('SILVER_TO_GOLD','gold.audit_freshness','overwrite',_t0,'FAILURE',
            error_message=str(_e))
    raise


## Proof: Mart Row Counts + Propensity Check

In [0]:
print('=== Gold Mart Measures (from local_gold_build.py) ===\n')

# peak_usage_patterns: utc_hour, local_hour, day_of_week, day_name, device_category, country, total_plays, total_minutes
print(f"peak_usage_patterns: {spark.table(GOLD['peak_usage_patterns']).count():,} rows")
print('  measures: utc_hour, local_hour, day_of_week, day_name, device_category, country, total_plays, total_minutes')
display(spark.sql(f"""
    SELECT utc_hour, local_hour, day_of_week, day_name,
           device_category, country, total_plays, total_minutes
    FROM {table_ref(GOLD['peak_usage_patterns'])}
    LIMIT 5
"""))

# subscription_propensity_segments: user_id, total_minutes, total_plays, session_count, genre_diversity,
#   skip_rate, distinct_devices, discovery_rate, volume_norm, device_diversity, engagement_score,
#   propensity_tier, tier, is_premium, country, is_high_conversion_propensity
print(f"\nsubscription_propensity_segments: {spark.table(GOLD['subscription_propensity_segments']).count():,} rows")
print('  measures: user_id, total_minutes, total_plays, session_count, genre_diversity, skip_rate, distinct_devices, discovery_rate, volume_norm, device_diversity, engagement_score, propensity_tier, tier, is_premium, country, is_high_conversion_propensity')
display(spark.sql(f"""
    SELECT user_id, total_minutes, total_plays, session_count, genre_diversity,
           skip_rate, distinct_devices, discovery_rate, volume_norm, device_diversity,
           engagement_score, propensity_tier, tier, is_premium, country,
           is_high_conversion_propensity
    FROM {table_ref(GOLD['subscription_propensity_segments'])}
    LIMIT 5
"""))

# content_performance_trends: year_month, year, month, group_type, group_name, monthly_minutes, monthly_share, mom_delta
print(f"\ncontent_performance_trends: {spark.table(GOLD['content_performance_trends']).count():,} rows")
print('  measures: year_month, year, month, group_type, group_name, monthly_minutes, monthly_share, mom_delta')
display(spark.sql(f"""
    SELECT year_month, CAST(SUBSTRING(CAST(year_month AS STRING),1,4) AS INT) AS year,
           CAST(SUBSTRING(CAST(year_month AS STRING),5,2) AS INT) AS month,
           group_type, group_name, monthly_minutes, monthly_share, mom_delta
    FROM {table_ref(GOLD['content_performance_trends'])}
    LIMIT 5
"""))

# wrapped_user_year: flat measures from local script
# Table uses nested structs; extract to match local script's flat column names
print(f"\nwrapped_user_year: {spark.table(GOLD['wrapped_user_year']).count():,} rows")
print('  measures: user_id, year, total_minutes, total_events, total_plays, skipped_plays, overall_skip_rate, source_skip_rate, distinct_tracks, distinct_artists, active_days, span_days, avg_minutes_per_active_day, longest_streak_days, streak_start, streak_end, biggest_day, biggest_day_minutes, biggest_day_plays, biggest_day_artist, biggest_day_artist_plays, most_replayed_track, most_replayed_artist, most_replayed_count, most_replayed_date, peak_hour, peak_hour_plays, quietest_hour, listening_age, listening_age_year, top_artist_1..5, top_genre_1..5, top_track_1..5, discovery_rate')
display(spark.sql(f"""
    SELECT user_id, year, total_minutes, total_events, total_plays,
           skip_count AS skipped_plays, skip_pct AS overall_skip_rate,
           distinct_tracks, distinct_artists, active_days, span_days,
           biggest_day.artist AS biggest_day_artist,
           biggest_day.minutes AS biggest_day_minutes,
           biggest_day.plays AS biggest_day_plays,
           biggest_day.artist_plays AS biggest_day_artist_plays,
           most_replayed.title AS most_replayed_track,
           most_replayed.artist AS most_replayed_artist,
           most_replayed.times AS most_replayed_count,
           longest_streak.length AS longest_streak_days,
           get(top_artists, 0).name AS top_artist_1,
           get(top_artists, 1).name AS top_artist_2,
           get(top_artists, 2).name AS top_artist_3,
           get(top_artists, 3).name AS top_artist_4,
           get(top_artists, 4).name AS top_artist_5,
           get(top_genres, 0).name AS top_genre_1,
           get(top_genres, 1).name AS top_genre_2,
           get(top_genres, 2).name AS top_genre_3,
           get(top_genres, 3).name AS top_genre_4,
           get(top_genres, 4).name AS top_genre_5,
           get(top_tracks, 0).title AS top_track_1,
           get(top_tracks, 1).title AS top_track_2,
           get(top_tracks, 2).title AS top_track_3,
           get(top_tracks, 3).title AS top_track_4,
           get(top_tracks, 4).title AS top_track_5,
           peak_hour, discovery_rate
    FROM {table_ref(GOLD['wrapped_user_year'])}
    ORDER BY total_minutes DESC LIMIT 5
"""))


## Audit Mart 4: Persisted Test Results (`audit_test_results`)

`tests/03_test_gold.ipynb` asserts and stops at the first failure; this mart persists
every check as a `PASS` / `FAIL` / `INFO` row so reports and dashboards can cite the
exact warehouse state. Mirrors `audit_test_results` from `scripts/local_gold_build.py`
(`run_checks` + `wrapped_checks`):

* **T1.1 / T1.2** fact vs silver parity, PK uniqueness
* **T2** referential integrity — one row per FK (user / track / device / date / hour)
* **T3.1 / T3.2** SCD2 `dim_user` integrity
* **T4.1** minutes conserved (fact vs silver)
* **T5** silver Delta version vs `ops.pipeline_state` (INFO)
* **T6** propensity match vs `validation/user_segments.json` ground truth (>= 80%)
* **T7.1** genre monthly shares sum to 1
* **W1 / W2 / W3 / W5, W9** Wrapped reconciliation against `fact_listening`

Failures are recorded, not raised — the point is an auditable snapshot.
Note: W3 uses the source `is_skipped` flag, which is NULL on some real-export rows;
NULLs are counted as plays in `total_plays` but excluded from the `hours` curve, so
W3 can legitimately FAIL while W2 passes (real finding, not a bug in the check).

In [0]:
import os, json

_t0 = datetime.now(timezone.utc)
try:
    checks = []

    def add(name, ok, detail=''):
        checks.append((name, 'PASS' if ok else 'FAIL', str(detail)))

    def info(name, detail):
        checks.append((name, 'INFO', str(detail)))

    def scalar(sql):
        row = spark.sql(sql).first()
        return None if row is None else row[0]

    # ---- T1: fact vs silver parity + PK uniqueness ----
    fact_n   = scalar(f"SELECT COUNT(*) FROM {table_ref(GOLD['fact_listening'])}")
    silver_n = scalar(f"SELECT COUNT(*) FROM {table_ref(SILVER_EVENTS)}")
    fact_d   = scalar(f"SELECT COUNT(DISTINCT event_id) FROM {table_ref(GOLD['fact_listening'])}")
    add('T1.1 fact rows == silver rows', fact_n == silver_n, f'{fact_n:,} vs {silver_n:,}')
    add('T1.2 fact event_id unique', fact_n == fact_d, f'{fact_n:,} rows, {fact_d:,} distinct ids')

    # ---- T2: referential integrity, one row per foreign key ----
    for fk, dim, key in (
            ('user_sk',     GOLD['dim_user'],   'user_sk'),
            ('track_sk',    GOLD['dim_track'],  'track_sk'),
            ('device_sk',   GOLD['dim_device'], 'device_sk'),
            ('date_key',    GOLD['dim_date'],   'date_key'),
            ('hour_of_day', GOLD['dim_time'],   'hour_of_day')):
        orphans = scalar(f"""
            SELECT COUNT(*) FROM (
                SELECT f.event_id FROM {table_ref(GOLD['fact_listening'])} f
                LEFT ANTI JOIN {table_ref(dim)} d ON f['{fk}'] = d['{key}'])
        """)
        add(f'T2 no orphan {fk}', orphans == 0, f'{orphans:,} orphans')

    # ---- T3: SCD2 integrity ----
    multi = scalar(f"""
        SELECT COUNT(*) FROM (
            SELECT user_id FROM {table_ref(GOLD['dim_user'])}
            WHERE is_current GROUP BY user_id HAVING COUNT(*) > 1)
    """)
    add('T3.1 one current row per user', multi == 0, f'{multi} users with multiple current rows')
    zero = scalar(f"""
        SELECT COUNT(*) FROM (
            SELECT user_id, MAX(CAST(is_current AS INT)) AS cur
            FROM {table_ref(GOLD['dim_user'])} GROUP BY user_id)
        WHERE cur = 0
    """)
    add('T3.2 no user without a current row', zero == 0, f'{zero} users with zero current rows')

    # ---- T4: metric conservation ----
    diff = scalar(f"""
        SELECT ABS(
            (SELECT COALESCE(SUM(minutes_played), 0) FROM {table_ref(GOLD['fact_listening'])})
          - (SELECT COALESCE(SUM(ms_played), 0) / 60000.0 FROM {table_ref(SILVER_EVENTS)}))
    """)
    add('T4.1 minutes conserved (fact == silver)', diff < 0.001, f'diff={diff:.6f}')

    # ---- T5: pipeline_state sync with the live silver version ----
    try:
        cur_v = spark.sql(f"DESCRIBE HISTORY {table_ref(SILVER_EVENTS)} LIMIT 1").first()['version']
        st = spark.table(PIPELINE_STATE).filter(f"table_name = '{SILVER_EVENTS}'").first()
        info('T5 silver version vs pipeline_state',
             f"state last_version={st['last_version']}, live silver version={cur_v}" if st
             else 'no pipeline_state entry for silver.events')
    except Exception as ex:
        info('T5 silver version vs pipeline_state', f'skipped: {ex}')

    # ---- T6: propensity match vs ground-truth personas (free users only) ----
    seg_path = os.path.join(LANDING_ROOT, 'validation', 'user_segments.json')
    if os.path.exists(seg_path):
        with open(seg_path, 'r', encoding='utf-8') as fh:
            ground_truth = json.load(fh)
        pred = {r.user_id: bool(r.is_high_conversion_propensity)
                for r in spark.table(GOLD['subscription_propensity_segments'])
                        .select('user_id', 'is_high_conversion_propensity').collect()}
        pairs = [(u, pred[u['user_id']]) for u in ground_truth
                 if not u.get('is_premium', False) and u['user_id'] in pred]
        matches = sum(1 for u, p in pairs
                      if (u.get('propensity_flag') == 'High Conversion Propensity') == p)
        rate = matches / max(len(pairs), 1)
        add('T6 propensity match vs ground truth (>= 80%)', rate >= 0.80,
            f'{rate:.1%} ({matches}/{len(pairs)}) free users')
    else:
        info('T6 propensity match vs ground truth', f'skipped: {seg_path} not found')

    # ---- T7: market-share completeness ----
    bad_ym = scalar(f"""
        SELECT COUNT(*) FROM (
            SELECT year_month, SUM(monthly_share) AS s
            FROM {table_ref(GOLD['content_performance_trends'])}
            WHERE group_type = 'genre' GROUP BY year_month)
        WHERE ABS(s - 1) > 0.001
    """)
    add('T7.1 genre monthly shares sum to 1', bad_ym == 0, f'{bad_ym} year-months off')

    # ---- Wrapped reconciliation (local_gold_build.py wrapped_checks) ----
    wdiff = scalar(f"""
        SELECT ABS(
            (SELECT COALESCE(SUM(total_minutes), 0) FROM {table_ref(GOLD['wrapped_user_year'])})
          - (SELECT COALESCE(SUM(minutes_played), 0) FROM {table_ref(GOLD['fact_listening'])}))
    """)
    add('W1 wrapped minutes == fact minutes', wdiff < 0.01, f'diff={wdiff:.6f}')

    bad_w2 = scalar(f"""
        SELECT COUNT(*) FROM {table_ref(GOLD['wrapped_user_year'])}
        WHERE total_plays + skip_count <> total_events
    """)
    add('W2 wrapped plays + skips == events', bad_w2 == 0, f'{bad_w2} user-years off')

    bad_w3 = scalar(f"""
        SELECT COUNT(*) FROM {table_ref(GOLD['wrapped_user_year'])}
        WHERE aggregate(hours, 0, (acc, x) -> acc + x) <> total_plays
    """)
    add('W3 wrapped 24-hour curve sums to total_plays', bad_w3 == 0, f'{bad_w3} user-years off')

    bad_w5 = scalar(f"""
        SELECT COUNT(*) FROM {table_ref(GOLD['wrapped_user_year'])}
        WHERE longest_streak.length > active_days OR active_days > span_days
    """)
    add('W5 wrapped streak <= active_days <= span_days', bad_w5 == 0, f'{bad_w5} user-years off')

    n_w = scalar(f"SELECT COUNT(*) FROM {table_ref(GOLD['wrapped_user_year'])}")
    info('W9 wrapped user-years', f'{n_w} user-years in wrapped_user_year')

    df_checks = (spark.createDataFrame(checks, schema=StructType([
                      StructField('test', StringType(), False),
                      StructField('result', StringType(), False),
                      StructField('detail', StringType(), True)]))
                  .withColumn('load_timestamp', F.current_timestamp()))
    n = df_checks.count()
    df_checks.write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
        .saveAsTable(GOLD['audit_test_results'])
    log_run('SILVER_TO_GOLD', 'gold.audit_test_results', 'overwrite', _t0, 'SUCCESS',
            rows_read=n, rows_inserted=n)
    fails = sum(1 for _, r, _ in checks if r == 'FAIL')
    print(f'audit_test_results: {n} checks, {fails} FAIL')
    display(df_checks.orderBy('test'))
except Exception as _e:
    log_run('SILVER_TO_GOLD', 'gold.audit_test_results', 'overwrite', _t0, 'FAILURE',
            error_message=str(_e))
    raise

## Audit Mart 5: Event Funnel (`audit_event_funnel`)

Per user-year: where do events go between Landing, Silver DQ and Wrapped?

* `bronze_events` — **distinct event_ids** stored in `bronze.events` (bronze is append-only
  and holds re-ingested duplicates plus pre-window history — 2024 rows and 2025 events
  before the current landing window — so bronze_events >= silver_events + quarantined is
  expected; ingest-level rejects live in `bronze.events_quarantine`, summarised in
  `audit_quality_summary`). Silver quarantine is attributed to a user-year by joining
  back to `bronze.events` on `event_id` — silver's quarantine table has no partition column.
* `silver_events` — rows that passed Silver DQ
* `null_ms_played_dropped_from_wrapped` — `real_api` rows Wrapped must exclude
* `skips_under_30s` / `qualified_plays` — the Wrapped 30-second play rule

Mirrors `audit_event_funnel` in `scripts/local_gold_build.py`.

In [0]:
_t0 = datetime.now(timezone.utc)
try:
    bronze_uy = spark.sql(f"""
        SELECT user_id, SUBSTRING(played_at, 1, 4) AS year,
               COUNT(DISTINCT event_id) AS bronze_events
        FROM {table_ref(BRONZE_EVENTS)}
        GROUP BY user_id, SUBSTRING(played_at, 1, 4)
    """)
    silver_uy = spark.sql(f"""
        SELECT user_id, CAST(YEAR(played_at) AS STRING) AS year,
               COUNT(*) AS silver_events,
               SUM(CASE WHEN ms_played IS NULL THEN 1 ELSE 0 END) AS null_ms_played_dropped_from_wrapped,
               SUM(CASE WHEN ms_played < 30000 THEN 1 ELSE 0 END) AS skips_under_30s,
               SUM(CASE WHEN ms_played >= 30000 THEN 1 ELSE 0 END) AS qualified_plays
        FROM {table_ref(SILVER_EVENTS)}
        GROUP BY user_id, CAST(YEAR(played_at) AS STRING)
    """)
    df_q = spark.sql(f"""
        SELECT e.user_id, SUBSTRING(e.played_at, 1, 4) AS year, q._dq_rule, q.event_id
        FROM {table_ref(SILVER_EVENTS_QUARANTINE)} q
        JOIN {table_ref(BRONZE_EVENTS)} e ON q.event_id = e.event_id
    """).groupBy('user_id', 'year').pivot('_dq_rule').agg(F.countDistinct('event_id'))
    for c in [c for c in df_q.columns if c not in ('user_id', 'year')]:
        df_q = df_q.withColumnRenamed(c, f'quarantined_{c}')

    df_funnel = (bronze_uy.join(silver_uy, ['user_id', 'year'], 'full_outer')
                  .join(df_q, ['user_id', 'year'], 'left'))
    count_cols = [c for c in df_funnel.columns if c not in ('user_id', 'year')]
    df_funnel = df_funnel.fillna(0, subset=count_cols)
    for c in count_cols:
        df_funnel = df_funnel.withColumn(c, F.col(c).cast('bigint'))

    ordered = (['user_id', 'year', 'bronze_events']
               + sorted(c for c in df_funnel.columns if c.startswith('quarantined_'))
               + ['silver_events', 'null_ms_played_dropped_from_wrapped',
                  'skips_under_30s', 'qualified_plays'])
    df_funnel = df_funnel.select(*ordered).withColumn('load_timestamp', F.current_timestamp())

    n = df_funnel.count()
    df_funnel.write.format('delta').mode('overwrite').option('overwriteSchema', 'true') \
        .saveAsTable(GOLD['audit_event_funnel'])
    log_run('SILVER_TO_GOLD', 'gold.audit_event_funnel', 'overwrite', _t0, 'SUCCESS',
            rows_read=n, rows_inserted=n)
    print(f'audit_event_funnel: {n:,} user-years')
    display(df_funnel.orderBy('user_id', 'year').limit(20))
except Exception as _e:
    log_run('SILVER_TO_GOLD', 'gold.audit_event_funnel', 'overwrite', _t0, 'FAILURE',
            error_message=str(_e))
    raise